<a href="https://colab.research.google.com/github/Suha2166/FrontENd/blob/main/Final_year_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upload Your Real CST Dataset
Use the cell below to upload your dataset (CSV format) directly into the environment. Once uploaded, we will parse it to train the model.

In [ ]:
from google.colab import files
import io
import pandas as pd

print("Please upload your CSV file containing the CST simulation data:")
uploaded = files.upload()

# Get the filename of the uploaded file
if uploaded:
    filename = list(uploaded.keys())[0]
    # Load into a dataframe
    real_df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"\nSuccessfully loaded '{filename}'!")
    display(real_df.head())
else:
    print("No file uploaded. Please run this cell again to upload your dataset.")

Please upload your CSV file containing the CST simulation data:


Saving antenna_s_parameters-v3.csv to antenna_s_parameters-v3 (2).csv

Successfully loaded 'antenna_s_parameters-v3 (2).csv'!


,Frequency,S11,yc,xc,ws,wg,wf,sw,mt,ls,lp,lf,il,h,dp
0,2.000,-0.159258,6.5,5.5,27.0,27.0,3.272,6.0,0.035,27.0,13.1199,10.0,4.2,1.524,18.554163
1,2.006,-0.158984,6.5,5.5,27.0,27.0,3.272,6.0,0.035,27.0,13.1199,10.0,4.2,1.524,18.554163
2,2.012,-0.158496,6.5,5.5,27.0,27.0,3.272,6.0,0.035,27.0,13.1199,10.0,4.2,1.524,18.554163
3,2.018,-0.158075,6.5,5.5,27.0,27.0,3.272,6.0,0.035,27.0,13.1199,10.0,4.2,1.524,18.554163
4,2.024,-0.157980,6.5,5.5,27.0,27.0,3.272,6.0,0.035,27.0,13.1199,10.0,4.2,1.524,18.554163


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### Map Your Dataset Columns
Update the variable names below to match the exact column names in your uploaded file so that the model trains correctly.

In [ ]:
# Change these string names to match your exact CSV headers!
TARGET_FREQUENCY_COLUMN = 'Frequency'
TARGET_S11_COLUMN = 'S11'

# The exact parameter columns detected in your uploaded CSV file (13 design parameters)
REAL_PARAM_COLS = ['yc', 'xc', 'ws', 'wg', 'wf', 'sw', 'mt', 'ls', 'lp', 'lf', 'il', 'h', 'dp']

# Use this cell to verify if the columns are correct
try:
    # Overwrite the placeholder dataset with your real dataset
    df = real_df.copy()
    param_cols = REAL_PARAM_COLS

    # Rename inputs to standard names for our training pipeline
    df = df.rename(columns={
        TARGET_FREQUENCY_COLUMN: 'Frequency',
        TARGET_S11_COLUMN: 'S11'
    })

    print("Column mapping completed successfully! Run the training cell below to train on your dataset.")
except NameError:
    print("Please upload your file in the step above first.")
except KeyError as e:
    print(f"Error: Column {e} not found in your uploaded file. Please double-check the column names.")

Column mapping completed successfully! Run the training cell below to train on your dataset.


### Dynamic Dataset Merger & Retraining
This cell scans your current directory for all uploaded versions of your CST parameter files, merges them into a single high-capacity training dataset, removes any duplicate entries, and reports the combined size. This allows the model to learn the patterns from all your iteration steps simultaneously.

In [ ]:
import os
import glob
import pandas as pd

# Look for any CSV files containing 'antenna_s_parameters' in the current workspace directory
csv_files = glob.glob('/content/antenna_s_parameters*.csv')

if len(csv_files) > 0:
    print(f"Found {len(csv_files)} datasets to merge:")
    for f in csv_files:
        print(f" - {os.path.basename(f)}")

    # Read and concatenate all detected datasets
    df_list = [pd.read_csv(f) for f in csv_files]
    merged_df = pd.concat(df_list, ignore_index=True)

    # Print initial row count
    initial_count = len(merged_df)

    # Remove exact duplicate simulation rows if any overlap exists
    merged_df = merged_df.drop_duplicates()
    final_count = len(merged_df)

    print(f"\nSuccess! Merged {initial_count} total rows into a clean dataset of {final_count} unique rows.")

    # Set this merged dataset as our active dataset
    real_df = merged_df
    df = real_df.copy()

    # Rename columns to standard names
    df = df.rename(columns={
        TARGET_FREQUENCY_COLUMN: 'Frequency',
        TARGET_S11_COLUMN: 'S11'
    })
    param_cols = REAL_PARAM_COLS

    display(df.head())
else:
    print("No uploaded datasets found. Please make sure your files are uploaded in the directory first.")

### 1. Generate Synthetic CST Dataset
Since your actual dataset is not yet loaded, we will generate a synthetic dataset with 1,000 samples containing:
* **Inputs for our ML Model**: `Frequency` and `S11` magnitude.
* **Targets for our ML Model**: 11 design parameters (`P1` to `P11`).

In [ ]:
import numpy as np
import pandas as pd

# Check if real dataset is already loaded via the uploader
try:
    if 'real_df' in globals():
        print("Using your uploaded real dataset (no synthetic data generated).")
        # Ensure we point df to your real data
        df = real_df.copy()
        df = df.rename(columns={
            TARGET_FREQUENCY_COLUMN: 'Frequency',
            TARGET_S11_COLUMN: 'S11'
        })
        param_cols = REAL_PARAM_COLS
        display(df.head())
    else:
        raise NameError
except NameError:
    print("No real dataset found. Generating synthetic dataset as placeholder...")
    # Fallback to synthetic data generation
    np.random.seed(42)
    n_samples = 1000
    parameters = np.random.rand(n_samples, 11)
    param_cols = [f'P{i}' for i in range(1, 12)]
    df = pd.DataFrame(parameters, columns=param_cols)
    df['Frequency'] = np.random.uniform(1.0, 10.0, n_samples)
    df['S11'] = -20 * np.sin(df['Frequency'] * df['P1']) - 5 * df['P2'] + np.random.normal(0, 1, n_samples)
    display(df.head())

,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,Frequency,S11
0,0.374540,0.950714,0.731994,0.598658,0.156019,0.155995,0.058084,0.866176,0.601115,0.708073,0.020584,5.797275,-21.880230
1,0.969910,0.832443,0.212339,0.181825,0.183405,0.304242,0.524756,0.431945,0.291229,0.611853,0.139494,2.241089,-20.406655
2,0.292145,0.366362,0.456070,0.785176,0.199674,0.514234,0.592415,0.046450,0.607545,0.170524,0.065052,6.321186,-20.297596
3,0.948886,0.965632,0.808397,0.304614,0.097672,0.684233,0.440152,0.122038,0.495177,0.034389,0.909320,3.833071,3.949898
4,0.258780,0.662522,0.311711,0.520068,0.546710,0.184854,0.969585,0.775133,0.939499,0.894827,0.597900,1.471139,-9.962686


### 2. Train Multi-Output Regression Model
We will map `['Frequency', 'S11']` to the 11 target parameters using a `RandomForestRegressor`, which natively handles multiple targets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Define Inputs (X) and Targets (y)
X = df[['Frequency', 'S11']]
y = df[param_cols]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the multi-output Random Forest regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate model performance
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred, multioutput='uniform_average')

print(f"Mean Absolute Error across all parameters: {mae:.4f}")
print(f"Average R² Score: {r2:.4f}")

Mean Absolute Error across all parameters: 0.2495
Average R² Score: -0.1218


### 3. Predict Parameters for a Target Frequency and S11
Now we define a helper function so you can query a specific target frequency and targeted S11 response to find the optimized parameters.

In [ ]:
def predict_design_parameters(target_frequency, target_s11):
    """
    Takes target performance requirements and predicts the physical parameters.
    """
    # Format input for model prediction
    input_data = np.array([[target_frequency, target_s11]])
    predicted_params = model.predict(input_data)[0]

    # Display predictions beautifully
    result_df = pd.DataFrame([predicted_params], columns=param_cols)
    print(f"\n--- Predicted Parameters for Freq: {target_frequency} GHz, S11: {target_s11} dB ---")
    display(result_df)
    return predicted_params

# Dynamic User Input
try:
    user_freq = float(input("Enter target frequency (e.g. 5.5): "))
    user_s11 = float(input("Enter target S11 (e.g. -15.0): "))

    # Run prediction
    predicted_list = predict_design_parameters(target_frequency=user_freq, target_s11=user_s11)
except ValueError:
    print("Invalid input! Please enter numerical values for both frequency and S11.")

--- Predicted Parameters for Freq: 5.5 GHz, S11: -15.0 dB ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11
0,0.233321,0.297951,0.413277,0.650574,0.403042,0.45012,0.477818,0.55215,0.573759,0.584238,0.265283
